# 05 — Choix des stations sous budget (Chiffrer + Prioriser)

**Rôle dans le pipeline :** avec un budget donné, choisir **où** installer les stations pour servir le plus de besoin possible, **sans compter deux fois les mêmes habitants**.

| Lit | Produit |
|---|---|
| `silver/emplacements.parquet` | `silver/selection.parquet` + `resultats/selection_stations.csv` |

**Pour la démo :** le jury peut proposer un autre budget → modifier `BUDGET_EUR` dans la cellule suivante, puis relancer **ce notebook seulement** (quelques secondes).

⚠️ `COUT_STATION_EUR` est **provisoire et non sourcé** : à remplacer par le chiffrage de l'équipe avant la soutenance.

In [1]:
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree
from parametres import (SILVER, RESULTATS_DIR, COMMUNES, RAYON_STATION_M, RAYON_TERRE_M,
                        BUDGET_EUR, COUT_STATION_EUR)

# Pour tester un autre budget pendant la démo : décommenter et modifier, puis relancer.
# BUDGET_EUR = 500_000

emplacements = pd.read_parquet(SILVER / "emplacements.parquet")
print(f"Budget : {BUDGET_EUR:,} € | coût d'une station : {COUT_STATION_EUR:,} € (provisoire)".replace(",", " "))

Budget : 300 000 € | coût d'une station : 30 000 € (provisoire)


### Méthode gloutonne (« greedy »)

À chaque tour :
1. calculer, pour chaque candidat, le **gain** = besoin des carreaux de sa zone **pas encore couverts** ;
2. retenir le candidat au plus grand gain **par euro** (avec un coût identique partout, c'est simplement le plus grand gain) ;
3. marquer ses carreaux comme couverts ;
4. recommencer tant que le budget le permet et qu'il reste du besoin à couvrir.

**Pourquoi cette méthode :** simple à expliquer, rapide (le jury peut changer le budget en direct), et c'est la méthode de référence pour ce type de problème (« couverture maximale ») : elle garantit au moins ~63 % du meilleur résultat possible, et fait en pratique bien mieux.

Le calcul par euro permettra plus tard des coûts **différents selon l'emplacement** (foncier, borne électrique…) : il suffira de remplir une colonne `cout_eur`.

In [2]:
def choisir_stations(emplacements, budget, cout_station, rayon_m):
    points = np.radians(emplacements[["lat", "lon"]].values)
    couverts = BallTree(points, metric="haversine").query_radius(points, r=rayon_m / RAYON_TERRE_M)
    besoin = emplacements["pop_ponderee"].values
    ind = emplacements["ind"].values
    cout = emplacements["cout_eur"].values if "cout_eur" in emplacements else np.full(len(emplacements), cout_station)

    deja_couvert = np.zeros(len(emplacements), dtype=bool)
    reste = budget
    choix = []
    while True:
        gains = np.array([besoin[v][~deja_couvert[v]].sum() for v in couverts])
        gains[cout > reste] = 0                 # trop cher pour le budget restant
        i = int(np.argmax(gains / cout))
        if gains[i] <= 0:
            break
        nouveaux = couverts[i][~deja_couvert[couverts[i]]]
        deja_couvert[nouveaux] = True
        reste -= cout[i]
        ligne = emplacements.iloc[i]
        choix.append({
            "rang": len(choix) + 1,
            "idcar_200m": ligne["idcar_200m"],
            "commune": COMMUNES.get(ligne["commune"], ligne["commune"]),
            "lat": ligne["lat"],
            "lon": ligne["lon"],
            "niveau_offre": ligne["niveau_offre"],
            "habitants_nouveaux": ind[nouveaux].sum(),
            "besoin_couvert": gains[i],
            "cout_eur": cout[i],
            "budget_restant_eur": reste,
        })
    return pd.DataFrame(choix), deja_couvert


selection, deja_couvert = choisir_stations(emplacements, BUDGET_EUR, COUT_STATION_EUR, RAYON_STATION_M)

### Résultat

- **Score /100** : gain de la station rapporté à celui de la **première** (la meilleure) ; il ne peut que baisser d'un rang au suivant.
- **Habitants nouvellement couverts** : chaque habitant n'est compté **qu'une fois**, pour la première station qui le couvre.

In [3]:
assert not selection.empty, "budget insuffisant pour une seule station"
selection["score_100"] = (100 * selection["besoin_couvert"] / selection["besoin_couvert"].iloc[0]).round()

pop = emplacements["ind"].sum()
besoin_total = emplacements["pop_ponderee"].sum()
print(f"{len(selection)} stations | {selection['cout_eur'].sum():,.0f} € engagés sur {BUDGET_EUR:,} €".replace(",", " "))
print(f"Habitants à moins de {RAYON_STATION_M} m d'une station : {selection['habitants_nouveaux'].sum():,.0f} "
      f"({selection['habitants_nouveaux'].sum() / pop:.1%} du territoire)".replace(",", " "))
print(f"Besoin couvert : {selection['besoin_couvert'].sum() / besoin_total:.1%} du besoin total")

selection[["rang", "score_100", "commune", "niveau_offre", "habitants_nouveaux", "besoin_couvert", "lat", "lon"]].round(3)

10 stations | 300 000 € engagés sur 300 000 €
Habitants à moins de 500 m d'une station : 32 544 (14.8% du territoire)
Besoin couvert : 18.9% du besoin total


,rang,score_100,commune,niveau_offre,habitants_nouveaux,besoin_couvert,lat,lon
0,1,100.0,Saint-Paul,Faible,2501.5,1545.55,-21.061,55.228
1,2,100.0,La Possession,Faible,4074.0,1544.40,-20.955,55.327
2,3,99.0,Saint-Paul,Aucun arrêt,2186.5,1526.90,-21.095,55.241
3,4,89.0,Saint-Paul,Correcte,3241.5,1378.35,-21.045,55.261
4,5,80.0,Le Port,Correcte,5048.5,1235.70,-20.935,55.306
5,6,71.0,Saint-Paul,Forte,4235.5,1099.35,-21.023,55.269
6,7,71.0,La Possession,Aucun arrêt,2373.5,1093.35,-20.959,55.343
7,8,70.0,Saint-Paul,Très faible,1084.0,1082.40,-21.084,55.237
8,9,66.0,Le Port,Forte,5754.5,1024.30,-20.944,55.291
9,10,64.0,Saint-Leu,Faible,2044.5,993.30,-21.197,55.298


In [4]:
RESULTATS_DIR.mkdir(parents=True, exist_ok=True)
selection.to_parquet(SILVER / "selection.parquet", index=False)
# Point-virgule et virgule décimale : le CSV s'ouvre directement dans Excel en français
selection.round(5).to_csv(RESULTATS_DIR / "selection_stations.csv", sep=";", decimal=",", index=False, encoding="utf-8-sig")
print("Écrit :", SILVER / "selection.parquet", "et", RESULTATS_DIR / "selection_stations.csv")

Écrit : C:\Users\nicol\Documents\Epitech\hackathon-citiz\data\silver\selection.parquet et C:\Users\nicol\Documents\Epitech\hackathon-citiz\resultats\selection_stations.csv
